# Start with hardware scan

### Find CPU number, Main Memory Size, Cache sizes (Apple Silicon)

In [2]:
import psutil
from cpuinfo import get_cpu_info

def bytes_to_mb(x):
    return f"{x / (1024 ** 2):.1f} MB"

def bytes_to_gb(x):
    return f"{x / (1024 ** 3):.2f} GB"

cpu = get_cpu_info()

# CPU cores
physical_cores = psutil.cpu_count(logical=False)
logical_cores = psutil.cpu_count(logical=True)

# RAM
ram = psutil.virtual_memory().total

# Cache info (may be missing on some CPUs/OSes)
l1 = cpu.get("l1_data_cache_size") or cpu.get("l1_cache_size")
l2 = cpu.get("l2_cache_size")
l3 = cpu.get("l3_cache_size")

print("CPU:")
print(f"  Brand           : {cpu.get('brand_raw', 'unknown')}")
print(f"  Physical cores  : {physical_cores}")
print(f"  Logical cores   : {logical_cores}")

print("\nMemory:")
print(f"  RAM             : {bytes_to_gb(ram)}")

print("\nCache:")
print(f"  L1 cache        : {bytes_to_mb(l1) if l1 else 'unknown'}")
print(f"  L2 cache        : {bytes_to_mb(l2) if l2 else 'unknown'}")
print(f"  L3 cache        : {bytes_to_mb(l3) if l3 else 'unknown'}")

CPU:
  Brand           : Intel(R) Core(TM) i7-7820HQ CPU @ 2.90GHz
  Physical cores  : 4
  Logical cores   : 8

Memory:
  RAM             : 31.20 GB

Cache:
  L1 cache        : 0.1 MB
  L2 cache        : 1.0 MB
  L3 cache        : 8.0 MB


### Find CPU number, Main Memory Size, Cache sizes (Generic)

In [3]:
import cpuinfo
import psutil

# ---- CPU Information ----
info = cpuinfo.get_cpu_info()

print("CPU Model:", info.get("brand_raw", "N/A"))

# Core counts
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores :", psutil.cpu_count(logical=True))

# ---- Memory ----
mem = psutil.virtual_memory()
print("Total RAM (GB):", round(mem.total / 1e9, 2))

# ---- Cache (if reported) ----
print("L2 Cache:", info.get("l2_cache_size", "N/A"))
print("L3 Cache:", info.get("l3_cache_size", "N/A"))

CPU Model: Intel(R) Core(TM) i7-7820HQ CPU @ 2.90GHz
Physical cores: 4
Logical cores : 8
Total RAM (GB): 33.5
L2 Cache: 1048576
L3 Cache: 8388608


# Timing primitive operators

In [4]:
import numpy as np
import time

x = np.random.rand(10**6)
y = np.random.rand(10**6)

k = 0.2

In [5]:
x.flags

  C_CONTIGUOUS : True
  F_CONTIGUOUS : True
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

In [6]:
%timeit x * y

1.2 ms ± 74.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
%timeit x / y

1.28 ms ± 31.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
%timeit x + y

1.22 ms ± 17.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%timeit x - y

1.2 ms ± 16.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
%timeit x / k

751 μs ± 9.89 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
%timeit x * (1/k)

765 μs ± 36.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%timeit x * k

754 μs ± 15.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [13]:
# ---- Performance Benchmark ----
def benchmark():
    size = 10**7
    a = np.random.rand(size)
    b = np.random.rand(size)

    start = time.time()
    c = a + b
    end = time.time()

    print("Time for vector addition (10M elements):", round(end - start, 4), "seconds") 

if __name__ == "__main__":
    benchmark()

Time for vector addition (10M elements): 0.0145 seconds


# Matrix Slicing

In [14]:
X = np.random.rand(20,20)
print(X.shape)
X.flags


(20, 20)


  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

In [15]:
X_sliced = X[3:5, 2:10]
print(X_sliced.shape)
X_sliced.flags

(2, 8)


  C_CONTIGUOUS : False
  F_CONTIGUOUS : False
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

In [16]:
print(np.asfortranarray(X_sliced).shape)
np.asfortranarray(X_sliced).flags

(2, 8)


  C_CONTIGUOUS : False
  F_CONTIGUOUS : True
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

In [17]:
print(np.ascontiguousarray(X_sliced).shape)
np.ascontiguousarray(X_sliced).flags

(2, 8)


  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

# Creating complex scalars

In [18]:
%timeit x = complex(8.1, 0.3)

107 ns ± 0.905 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [19]:
%timeit x = 8.1 + 1j*0.3

12 ns ± 0.53 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


# Using efficient numerical expression functions

In [22]:
import numexpr as ne
import numpy as np
ne.nthreads          # current number of threads
ne.detect_number_of_cores()
ne.set_num_threads(8)   # choose a value


a = np.arange(7E6)
b = np.arange(7E6)

%timeit a**2 + b**2 + 2*a*b
%timeit ne.evaluate("a**2 + b**2 + 2*a*b")

%timeit a**10 + a**7 + a**2*b**3
%timeit ne.evaluate("a**10 + a**7 + a**2*b**3")

47.6 ms ± 1.91 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
15 ms ± 173 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
440 ms ± 5.02 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
16.1 ms ± 125 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [2]:
import numpy as np
from numba import njit, prange, set_num_threads, get_num_threads

set_num_threads(8)      # set number of threads
print(get_num_threads())

a = np.arange(7_000_000, dtype=np.float64)
b = np.arange(7_000_000, dtype=np.float64)

@njit(parallel=True)
def expr1(a, b):
    out = np.empty_like(a)
    for i in prange(a.size):
        out[i] = a[i]**2 + b[i]**2 + 2*a[i]*b[i]
    return out

@njit(parallel=True)
def expr2(a, b):
    out = np.empty_like(a)
    for i in prange(a.size):
        out[i] = a[i]**10 + a[i]**7 + a[i]**2 * b[i]**3
    return out

# # ---- warm-up (compilation) ----
# _ = expr1(a, b)
# _ = expr2(a, b)

# ---- timing (like your numexpr example) ----
%timeit expr1(a, b)
%timeit expr2(a, b)


8
13.8 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
18.3 ms ± 2.68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Vectorization

In [3]:
import time
import numpy as np


def scalar(u, v):
    t0 = time.time()
    R = 0.0
    for n in range(len(u)):
        R += u[n] * v[n]
    dt = time.time() - t0
    return R, dt


def vectorized(u, v):
    t0 = time.time()
    R = np.dot(u.T, v)
    dt = time.time() - t0
    return R, dt


# ----------------------------------
# Testbench
# ----------------------------------
if __name__ == "__main__":

    N = int(10e6)

    u = np.random.normal(size=N)
    v = np.random.normal(size=N)

    # Scalar version
    x_sum_scalar, dt_scalar = scalar(u, v)
    print('%s\nScalar\n%s' % ('='*79, '='*79))
    print('Time: %20.6f [s]' % dt_scalar)
    print('Sum:  %20.6f' % x_sum_scalar)

    # Vectorized version
    x_sum_vectorized, dt_vectorized = vectorized(u, v)
    print('%s\nVectorized\n%s' % ('='*79, '='*79))
    print('Time: %20.6f [s]' % dt_vectorized)
    print('Sum:  %20.6f' % x_sum_vectorized)

    # ----------------------------------
    # Speedup
    # ----------------------------------
    speedup = dt_scalar / dt_vectorized
    print('%s\nPerformance\n%s' % ('='*79, '='*79))
    print('Speedup (Scalar / Vectorized): %10.2f x' % speedup)


Scalar
Time:             3.011660 [s]
Sum:            458.287975
Vectorized
Time:             0.006115 [s]
Sum:            458.287975
Performance
Speedup (Scalar / Vectorized):     492.47 x
